# Data Wrangling on Real Estate Market

## 1. Import Dataset and Clean Column Names

In [1]:
import pandas as pd
import numpy as np

# Using a placeholder link for the Real Estate dataset. Replace this URL with your specific dataset link if needed.
url = 'https://raw.githubusercontent.com/datasets/house-prices-uk/master/data/data.csv'

try:
    df = pd.read_csv(url)
except Exception:
    # Fallback to local file if URL fetch fails
    df = pd.read_csv('RealEstate_Prices.csv')

df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('[^a-zA-Z0-9_]', '', regex=True)

display(df.head())

,date,price_all,change_all,price_new,change_new,price_modern,change_modern,price_older,change_older
0,1953-02-01,1891.0,NaN,2107.0,NaN,2002.0,NaN,1542.0,NaN
1,1953-05-01,1891.0,NaN,2107.0,NaN,2002.0,NaN,1542.0,NaN
2,1953-08-01,1881.0,NaN,2117.0,NaN,2002.0,NaN,1524.0,NaN
3,1953-11-01,1872.0,-1.0,2117.0,0.5,1975.0,-2.2,1542.0,1.2
4,1954-02-01,1863.0,-1.5,2117.0,0.5,1957.0,-2.2,1524.0,-1.2


## 2. Handle Missing Values

In [2]:
print("Missing values per column:\n", df.isnull().sum())

for col in df.select_dtypes(include=[np.number]).columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].fillna(df[col].mode()[0])

print("Remaining missing values:", df.isnull().sum().sum())

Missing values per column:
 date              0
price_all         0
change_all        3
price_new         0
change_new        3
price_modern     39
change_modern    43
price_older      39
change_older     43
dtype: int64
Remaining missing values: 0


C:\Users\anjal\AppData\Local\Temp\ipykernel_12956\2085534656.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


## 3. Filter and Subset Data

In [3]:
if 'property_type' in df.columns:
    subset_df = df[df['property_type'].isin(['Residential', 'Single Family'])]
else:
    subset_df = df.copy()

display(subset_df.head())

,date,price_all,change_all,price_new,change_new,price_modern,change_modern,price_older,change_older
0,1953-02-01,1891.0,6.6,2107.0,6.2,2002.0,7.0,1542.0,7.7
1,1953-05-01,1891.0,6.6,2107.0,6.2,2002.0,7.0,1542.0,7.7
2,1953-08-01,1881.0,6.6,2117.0,6.2,2002.0,7.0,1524.0,7.7
3,1953-11-01,1872.0,-1.0,2117.0,0.5,1975.0,-2.2,1542.0,1.2
4,1954-02-01,1863.0,-1.5,2117.0,0.5,1957.0,-2.2,1524.0,-1.2


## 4. Categorical Variable Encoding

In [4]:
categorical_cols = subset_df.select_dtypes(include=['object']).columns.tolist()
print("Categorical columns to encode:", categorical_cols)

encoded_df = pd.get_dummies(subset_df, columns=categorical_cols, drop_first=True)
display(encoded_df.head())

Categorical columns to encode: ['date']


C:\Users\anjal\AppData\Local\Temp\ipykernel_12956\1814991332.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = subset_df.select_dtypes(include=['object']).columns.tolist()


,price_all,change_all,price_new,change_new,price_modern,change_modern,price_older,change_older,date_1953-05-01,date_1953-08-01,...,date_2023-02-01,date_2023-05-01,date_2023-08-01,date_2023-11-01,date_2024-02-01,date_2024-05-01,date_2024-08-01,date_2024-11-01,date_2025-02-01,date_2025-05-01
0,1891.0,6.6,2107.0,6.2,2002.0,7.0,1542.0,7.7,False,False,...,False,False,False,False,False,False,False,False,False,False
1,1891.0,6.6,2107.0,6.2,2002.0,7.0,1542.0,7.7,True,False,...,False,False,False,False,False,False,False,False,False,False
2,1881.0,6.6,2117.0,6.2,2002.0,7.0,1524.0,7.7,False,True,...,False,False,False,False,False,False,False,False,False,False
3,1872.0,-1.0,2117.0,0.5,1975.0,-2.2,1542.0,1.2,False,False,...,False,False,False,False,False,False,False,False,False,False
4,1863.0,-1.5,2117.0,0.5,1957.0,-2.2,1524.0,-1.2,False,False,...,False,False,False,False,False,False,False,False,False,False


## 5. Aggregate Data for Summary Statistics

In [5]:
if 'neighborhood' in df.columns and 'sale_price' in df.columns:
    neighborhood_summary = df.groupby('neighborhood')['sale_price'].agg(['mean', 'median', 'count']).reset_index()
    display(neighborhood_summary)
else:
    print("Required columns for neighborhood aggregation are missing.")

Required columns for neighborhood aggregation are missing.


## 6. Outlier Detection and Handling

In [6]:
if 'sale_price' in df.columns:
    Q1 = df['sale_price'].quantile(0.25)
    Q3 = df['sale_price'].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    cleaned_df = df[(df['sale_price'] >= lower_bound) & (df['sale_price'] <= upper_bound)]
    print(f"Original shape: {df.shape}, Cleaned shape: {cleaned_df.shape}")
else:
    print("Sale price column not found for outlier removal.")

Sale price column not found for outlier removal.
